In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.features import build_supervised_dataset
from src.train import temporal_split

# Recreate the same data as the sample_prices fixture
start = pd.Timestamp("2024-01-01", tz="Europe/Athens")
end = pd.Timestamp("2024-04-01", tz="Europe/Athens")
idx = pd.date_range(start=start, end=end, freq="h", inclusive="left")

rng = np.random.default_rng(seed=42)
hour = idx.hour.to_numpy()
dow = idx.dayofweek.to_numpy()

base = 100.0
daily = 30 * np.sin((hour - 7) * np.pi / 12)
weekly = np.where(dow >= 5, -15.0, 0.0)
noise = rng.normal(0, 5, len(idx))

values = base + daily + weekly + noise
sample_prices = pd.Series(values, index=idx, name="price_eur_mwh")

# Now use it normally
X, y, meta = build_supervised_dataset(sample_prices)
_, _, _, _, _, m_te = temporal_split(X, y, meta, test_days=14)

counts = m_te.groupby("forecast_time").size()
print(counts)
print(f"\nDays with fewer than 24 horizons: {(counts < 24).sum()}")
print(f"Total rows: {counts.sum()}")
print(f"Number of forecast days: {len(counts)}")

forecast_time
2024-03-17 12:00:00+02:00    24
2024-03-18 12:00:00+02:00    24
2024-03-19 12:00:00+02:00    24
2024-03-20 12:00:00+02:00    24
2024-03-21 12:00:00+02:00    24
2024-03-22 12:00:00+02:00    24
2024-03-23 12:00:00+02:00    24
2024-03-24 12:00:00+02:00    24
2024-03-25 12:00:00+02:00    24
2024-03-26 12:00:00+02:00    24
2024-03-27 12:00:00+02:00    24
2024-03-28 12:00:00+02:00    24
2024-03-29 12:00:00+02:00    24
2024-03-30 12:00:00+02:00    23
dtype: int64

Days with fewer than 24 horizons: 1
Total rows: 335
Number of forecast days: 14


In [11]:
meta[meta['forecast_time'] == '2024-03-30 12:00:00+02:00']

,forecast_time,horizon,target_time
89,2024-03-30 12:00:00+02:00,0,2024-03-31 00:00:00+02:00
179,2024-03-30 12:00:00+02:00,1,2024-03-31 01:00:00+02:00
269,2024-03-30 12:00:00+02:00,2,2024-03-31 02:00:00+02:00
359,2024-03-30 12:00:00+02:00,3,2024-03-31 04:00:00+03:00
449,2024-03-30 12:00:00+02:00,4,2024-03-31 05:00:00+03:00
539,2024-03-30 12:00:00+02:00,5,2024-03-31 06:00:00+03:00
629,2024-03-30 12:00:00+02:00,6,2024-03-31 07:00:00+03:00
719,2024-03-30 12:00:00+02:00,7,2024-03-31 08:00:00+03:00
809,2024-03-30 12:00:00+02:00,8,2024-03-31 09:00:00+03:00
899,2024-03-30 12:00:00+02:00,9,2024-03-31 10:00:00+03:00


In [ ]:
# Throwaway snippet to inspect what pd.date_range does
idx = pd.date_range(
    pd.Timestamp("2024-10-27", tz="Europe/Athens"),
    pd.Timestamp("2024-10-28", tz="Europe/Athens"),
    freq="h",
    inclusive="left",
)
print(len(idx))           # should be 25
print(idx)